# Visualization Module

Produces **5 charts** (4 required + 1 bonus) from cleaned price data.

| # | Chart | Type |
|---|-------|------|
| 1 | Price trend + volume overlay | Interactive line + bar (Plotly) |
| 2 | Correlation heatmap of daily returns | Seaborn heatmap |
| 3 | Distribution of daily returns | Violin / KDE (Plotly) |
| 4 | Bollinger Bands (rolling statistics) | Interactive line + fill (Plotly) |
| 5 | Normalised performance comparison | Multi-line indexed (Plotly) |

HTML charts are saved to `outputs/charts/`. Static charts saved as PNG.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings("ignore")
os.makedirs("outputs/charts", exist_ok=True)

# Open Plotly charts in the browser (avoids nbformat dependency)
pio.renderers.default = "browser"

# ── Tickers to visualise ────────────────────────────────────────────────────
FOCUS_TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "JPM", "JNJ"]
CHART_TICKER  = "AAPL"   # single-ticker charts (Chart 1 & 4)
BB_TICKER     = "NVDA"   # Bollinger Bands ticker
START_2Y      = "2024-01-01"   # window for 2-year views

print("Visualization Module")
print("=" * 50)
print(f"Focus tickers : {FOCUS_TICKERS}")
print(f"Price trend   : {CHART_TICKER}")
print(f"Bollinger     : {BB_TICKER}")
print(f"Plotly renderer: {pio.renderers.default}")

In [ ]:
# ── Load cleaned price data ─────────────────────────────────────────────────
price_df = pd.read_csv("data/cleaned/prices_clean.csv", parse_dates=["Date"])
price_df = price_df[price_df["Ticker"].isin(FOCUS_TICKERS)].copy()
price_df = price_df.sort_values(["Ticker", "Date"])

price_2y = price_df[price_df["Date"] >= START_2Y].copy()

print(f"Loaded  : {price_df.shape[0]:,} rows | {price_df['Ticker'].nunique()} tickers")
print(f"Range   : {price_df['Date'].min().date()} → {price_df['Date'].max().date()}")
print(f"2-year  : {price_2y['Date'].min().date()} → {price_2y['Date'].max().date()}")

## Chart 1 — Price Trend with Volume Overlay

In [ ]:
df1 = price_2y[price_2y["Ticker"] == CHART_TICKER].copy()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.75, 0.25],
    subplot_titles=(f"{CHART_TICKER} — Close Price & Moving Averages", "Volume"),
)

fig.add_trace(go.Scatter(
    x=df1["Date"], y=df1["Close"],
    name="Close", line=dict(color="#1f77b4", width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df1["Date"], y=df1["ma7"],
    name="7-day MA", line=dict(color="#ff7f0e", width=1.2, dash="dot")
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df1["Date"], y=df1["ma30"],
    name="30-day MA", line=dict(color="#2ca02c", width=1.8)
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df1["Date"], y=df1["Volume"],
    name="Volume", marker_color="rgba(100,100,200,0.4)"
), row=2, col=1)

fig.update_layout(
    title=f"{CHART_TICKER} — Price Trend with Volume Overlay (2024–2025)",
    height=600, template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    yaxis_title="Price (USD)",
    yaxis2_title="Volume",
    xaxis2_title="Date",
)

out = "outputs/charts/chart1_price_trend.html"
fig.write_html(out)
fig.show()
print(f"Saved: {out}")

## Chart 2 — Correlation Heatmap of Daily Returns

returns_wide = (
    price_2y
    .pivot_table(index="Date", columns="Ticker", values="daily_return")
    .dropna(how="all")
)
corr = returns_wide.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))   # hide upper triangle

fig2, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, mask=mask,
    annot=True, fmt=".2f",
    cmap="RdYlGn", vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    cbar_kws={"label": "Pearson Correlation"},
)
ax.set_title("Correlation of Daily Returns — Focus Tickers (2024–2025)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()

out = "outputs/charts/chart2_correlation_heatmap.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

## Chart 3 — Distribution of Daily Returns (Violin Plot)

In [ ]:
fig3 = go.Figure()
for ticker in FOCUS_TICKERS:
    ret = price_df[price_df["Ticker"] == ticker]["daily_return"].dropna()
    fig3.add_trace(go.Violin(
        x=[ticker] * len(ret), y=ret,
        name=ticker,
        box_visible=True,
        meanline_visible=True,
        points="outliers",
    ))

fig3.update_layout(
    title="Distribution of Daily Returns — Focus Tickers (Full History)",
    xaxis_title="Ticker",
    yaxis_title="Daily Return",
    yaxis=dict(tickformat=".1%"),
    template="plotly_white",
    height=550,
    showlegend=False,
)

out = "outputs/charts/chart3_return_distribution.html"
fig3.write_html(out)
fig3.show()
print(f"Saved: {out}")

## Chart 4 — Bollinger Bands (Rolling Statistics)

In [ ]:
df4 = price_2y[price_2y["Ticker"] == BB_TICKER].dropna(subset=["bb_upper", "bb_lower"]).copy()

fig4 = go.Figure()

# Shaded band fill
fig4.add_trace(go.Scatter(
    x=df4["Date"], y=df4["bb_upper"],
    name="Upper Band (2σ)",
    line=dict(color="rgba(173,173,220,0.5)", width=1),
    mode="lines",
))
fig4.add_trace(go.Scatter(
    x=df4["Date"], y=df4["bb_lower"],
    name="Lower Band (2σ)",
    line=dict(color="rgba(173,173,220,0.5)", width=1),
    fill="tonexty", fillcolor="rgba(173,173,220,0.2)",
    mode="lines",
))
# Midline (20-day MA)
fig4.add_trace(go.Scatter(
    x=df4["Date"], y=df4["bb_mid"],
    name="20-day MA",
    line=dict(color="#ff7f0e", width=1.5, dash="dot"),
))
# Close price
fig4.add_trace(go.Scatter(
    x=df4["Date"], y=df4["Close"],
    name="Close",
    line=dict(color="#1f77b4", width=2),
))

fig4.update_layout(
    title=f"{BB_TICKER} — Bollinger Bands (20-day, ±2σ) 2024–2025",
    xaxis_title="Date",
    yaxis_title="Price (USD)",
    template="plotly_white",
    height=550,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)

out = "outputs/charts/chart4_bollinger_bands.html"
fig4.write_html(out)
fig4.show()
print(f"Saved: {out}")